In [4]:
%pip install faiss-cpu
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableSequence, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredPowerPointLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_community.vectorstores import FAISS 
import os
import time
from dotenv import load_dotenv

load_dotenv()

chat_gemini_api_key = os.getenv('chat_gemini_api_key')
chat_gemini = ChatGoogleGenerativeAI(model='gemma-3-27b-it', api_key=chat_gemini_api_key)
embedding_gemini_model = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2-preview', 
    api_key=chat_gemini_api_key, 
    task_type="retrieval_document"
)

str_parser = StrOutputParser()

input_prompt = """
Task 2.2 — Parallel evidence gathering
Given a business KPI question, build a RunnableParallel that fires three sub-chains concurrently: one retrieves relevant operational context from a FAISS vector store, one searches for the metric definition using a mock tool, and one runs a simpler "quick answer" LLM pass. A final synthesis chain takes all three outputs and produces a single structured response. Measure and log the time saved vs running them sequentially.
"""

# 1. Read the documents: ppt, pdf, txt - DONE
# 2. Chunk the content of documents - DONE
# 3. store them in vectore store - DONE

# Create a series chain
# 1. Retrieves from vector database
# 2. Retrieves another from vector database
# 3. Create answer from the first questions
# 4. Create answer from the second question
# 5. Combines answer from step 3 and 4 to produce a single final output.

# Create a parallel chain
# 1. Retrieves from vector database -> Create answer from the first questions
# 2. Retrieves another from vector database -> Create answer from the second question
# 5. Combines answer from step 3 and 4 to produce a single final output.

# Finally compare the time of execution for both queries.


data_files_dir = []
cwd = os.getcwd()
data_files_dir = 'files'
for _, _, i in os.walk(data_files_dir):
    data_files_dir = list(map(lambda x: os.path.join(cwd, data_files_dir, x), i))

# print(data_files_dir)
txt_files = [file_path for file_path in data_files_dir if 'txt' in file_path ]
# print(txt_files)
ppt_files = [file_path for file_path in data_files_dir if 'ppt' in file_path ]
pdf_files = [file_path for file_path in data_files_dir if 'pdf' in file_path ]

## Read unstructured data and create Documents
txt_document = []
for file_path in txt_files:
    txt_loader = TextLoader(file_path)
    txt_document.extend(txt_loader.load())
# print(txt_document)
print('Length of txt document: ', len(txt_document))

pdf_document = []
for file_path in pdf_files:
    pdf_loader = PyPDFLoader(file_path)
    pdf_document.extend(pdf_loader.load())
# print(pdf_document)
print('Length of pdf document: ', len(pdf_document))

ppt_document = []
for file_path in ppt_files:
    ppt_loader = UnstructuredPowerPointLoader(file_path)
    ppt_document.extend(ppt_loader.load())
# print(ppt_document)
print('Length of ppt document: ', len(txt_document))

## Split the texts into multiple chunks based on charcater size and overlap size
rec_char_txt_splitter_obj = RecursiveCharacterTextSplitter(chunk_size=128, chunk_overlap=16, separators=['\n\n', '\n', ' '])
split_txt_documents = rec_char_txt_splitter_obj.split_documents(txt_document)
print(f'Length of txt_documents: {len(txt_document)}, length of split txt docuents: {len(split_txt_documents)}')

split_pdf_documents = rec_char_txt_splitter_obj.split_documents(pdf_document)
print(f'Length of txt_documents: {len(pdf_document)}, length of split txt docuents: {len(split_pdf_documents)}')

split_ppt_documents = rec_char_txt_splitter_obj.split_documents(ppt_document)
print(f'Length of txt_documents: {len(ppt_document)}, length of split txt docuents: {len(split_ppt_documents)}')

# for document in split_pdf_documents:
#     print(document.page_content, end='\n\n')

vectorstore = FAISS.from_documents(
    documents=split_txt_documents + split_pdf_documents + split_ppt_documents,
    embedding=embedding_gemini_model
)

retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k':5, 'lambda_mult':1})
res = retriever.invoke('What is gradient descent?')
print(res)

Length of txt document:  1
Length of pdf document:  1
Length of ppt document:  1
Length of txt_documents: 1, length of split txt docuents: 47
Length of txt_documents: 1, length of split txt docuents: 14
Length of txt_documents: 1, length of split txt docuents: 17
[Document(id='cd5ca1b7-0c7c-491b-8be8-ae9dd2fbe5e3', metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-15T12:53:12+05:30', 'author': 'Rohit Sharma', 'moddate': '2026-03-15T12:53:12+05:30', 'source': 'c:\\Python Practice\\langchain-notes\\files\\gradient_descent.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Gradient descent is a fundamental iterative optimization algorithm used in machine learning'), Document(id='fff1016e-1582-464b-8dfa-3efe8f6dd029', metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-15T12:53:12+05:30', 'author': 'Rohit Sharma', 'moddate': '2026-03-15T12:53:12+05:30', 'source': 

In [ ]:
print(len(res))

for document in res:
    print(document.page_content)

5
Gradient descent is a fundamental iterative optimization algorithm used in machine learning
small steps in the opposite direction, converging towards the minimum value. 
 
Key Concepts of Gradient Descent:
biases). It works by calculating the steepest downhill direction (the gradient) and taking
• Process: Starts with random parameter values and iteratively updates them:  
Main Types of Gradient Descent:
ascent; therefore, moving in the negative gradient direction leads to the minimum.


In [ ]:
# Create a series chain
# 1. Retrieves from vector database
# 2. Retrieves another from vector database
# 3. Create answer from the first question
# 4. Create answer from the second question
# 5. Combines answer from step 3 and 4 to produce a single final output. - DONE

first_retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k':5, 'lambda_mult':1})
second_retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k':5, 'lambda_mult':1})

# prompt_question_1 = ChatPromptTemplate.from_messages([
#     ('system', 'Helpful ai assistant helps in writing question and answers in concise manner with correct grammer and professional tone.'),
#     ('human', 'answer using the follwoing content question: {query_1} and answer: {result_1}')
# ]) 
# prompt_question_2 = ChatPromptTemplate.from_messages([
#     ('system', 'Helpful ai assistant helps in writing question and answers in concise manner with correct grammer and professional tone.'),
#     ('human', 'answer using the follwoing content question: {query_2} and answer: {result_2}')
# ]) 

# As gemma doesn't supports system messages, you need to use human level messages only when sending prompt to gemma.
prompt_question_1 = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template(
        'You are a helpful ai assistant helps in writing question and answers in concise manner with correct grammer and professional tone.Answer using the follwoing content question: {query_1} and answer: {result_1}'
    )
])
prompt_question_2 = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template(
        'you are a helpful ai assistant helps in writing question and answers in concise manner with correct grammer and professional tone. Answer using the follwoing content question: {query_2} and answer: {result_2}'
    )
])

sequential_chain = RunnableSequence(
    RunnableLambda(lambda x: {
        'result_1': first_retriever.invoke(x['query_1']),
        'query_2': x['query_2'],
        'query_1': x['query_1'],
    }),
    RunnableLambda(lambda x: {
        'result_2': second_retriever.invoke(x['query_2']),
        'result_1': x['result_1'],
        'query_1': x['query_1'],
        'query_2': x['query_2'],
    }),
    RunnableLambda(lambda x:{
        'result_output_1': prompt_question_1.invoke({'query_1': x['query_1'], 'result_1': x['result_1']}),
        'result_output_2': prompt_question_2.invoke({'query_2': x['query_2'], 'result_2': x['result_2']}),
        'query_1': x['query_1'],
        'query_2': x['query_2'],
        'result_1': x['result_1'],
        'result_2': x['result_2'],
    }),
    RunnableLambda(lambda x:{
        'output_1': chat_gemini.invoke(x['result_output_1']),
        'output_2': chat_gemini.invoke(x['result_output_2']),
    }),
    RunnableLambda(lambda x:{
        'final_output_1': str_parser.invoke(x['output_1']),
        'final_output_2': str_parser.invoke(x['output_2']),
    }),
)
    

In [66]:

start_time = time.time()
res = sequential_chain.invoke({'query_1': 'what is gradient descent?', 'query_2': 'What is decision tree?'})
print(res)
end_time = time.time()
print(f'Total time taken for execution via sequntial chain: {end_time-start_time}')

{'final_output_1': '**Question:** What is gradient descent?\n\n**Answer:** Gradient descent is a fundamental iterative optimization algorithm used in machine learning. It identifies the steepest downhill direction (the gradient) of a function—typically a loss function—and adjusts model parameters (weights and biases) in small steps in the opposite direction to converge towards the minimum value. Essentially, it minimizes the function by iteratively moving towards the lowest point.', 'final_output_2': '**Question:** What is a decision tree?\n\n**Answer:** A decision tree is a supervised machine learning algorithm utilizing a hierarchical, flowchart-like structure of sequential decisions to classify or predict outcomes. It is composed of nodes connected by branches, representing the outcomes of those decisions.'}
Total time taken for execution via sequntial chain: 7.622498512268066


In [ ]:
# Just make everything parallel. -LOL

parallel_chain = RunnableParallel({
    'branch_1': RunnableSequence(
        RunnableLambda(lambda x: {
            'result_1': first_retriever.invoke(x['query_1']),
            'query_1': x['query_1'],
        }),
        RunnableLambda(lambda x:{
            'result_output': prompt_question_1.invoke({'query_1': x['query_1'], 'result_1': x['result_1']}),
            'query': x['query_1'],
            'result': x['result_1'],
        }),
        RunnableLambda(lambda x:{
            'output': chat_gemini.invoke(x['result_output']),
        }),
        RunnableLambda(lambda x:{
            'final_output_1': str_parser.invoke(x['output']),
        }),
    ),

    'branch_2': RunnableSequence(
        RunnableLambda(lambda x: {
            'result_2': second_retriever.invoke(x['query_2']),
            'query_2': x['query_2'],
        }),
        RunnableLambda(lambda x:{
            'result_output': prompt_question_2.invoke({'query_2': x['query_2'], 'result_2': x['result_2']}),
            'query': x['query_2'],
            'result': x['result_2'],
        }),
        RunnableLambda(lambda x:{
            'output': chat_gemini.invoke(x['result_output']),
        }),
        RunnableLambda(lambda x:{
            'final_output_2': str_parser.invoke(x['output']),
        }),
    )
})

seq_parallel_chain = RunnableSequence(
    parallel_chain,
    RunnableParallel({
        'branch_1': RunnableLambda(lambda x: str_parser.invoke(x['branch_1']['final_output_1'])),
        'branch_2': RunnableLambda(lambda x: str_parser.invoke(x['branch_2']['final_output_2'])),
    })
)
# res = parallel_chain.invoke({'query_1': 'what is gradient descent?', 'query_2': 'What is decision tree?'})


In [76]:
start_time = time.time()
res = seq_parallel_chain.invoke({'query_1': 'what is gradient descent?', 'query_2': 'What is decision tree?'})
print(res)
end_time = time.time()
print(f'Total time taken for execution via parallel and sequential chain combination: {end_time-start_time}')

{'branch_1': '**Question:** What is gradient descent?\n\n**Answer:** Gradient descent is a fundamental iterative optimization algorithm used in machine learning. It identifies the steepest downhill direction (the gradient) of a function—typically a loss function—and adjusts model parameters (weights and biases) in small steps in the *opposite* direction to converge towards the minimum value, thereby optimizing the model.', 'branch_2': '**Question:** What is a decision tree?\n\n**Answer:** A decision tree is a supervised machine learning algorithm utilizing a hierarchical, flowchart-like structure of sequential decisions to classify or predict outcomes. It is composed of nodes connected by branches, representing the outcomes of those decisions.'}
Total time taken for execution via parallel and sequential chain combination: 3.3470335006713867
